# ELIVA25 Medical Bone Age - RunPod Edition

## Setup Instructions:
1. On RunPod, select a **GPU pod** (RTX 3090/4090/A6000 recommended) with **PyTorch or TensorFlow template**
2. Upload this notebook and open it in Jupyter
3. Set up your Kaggle API key (Cell 1)
4. Run all cells sequentially
5. The final cell saves `eliva_ensemble_submission.csv` — download it and submit manually on Kaggle

**Note:** Training takes ~5-10 hours depending on GPU. Close the browser — RunPod keeps it running.

In [ ]:
# Cell 1: Install dependencies (run once)
!pip install -q tensorflow kagglehub kaggle pandas numpy scikit-learn matplotlib

In [ ]:
# Cell 2: Set up Kaggle API credentials
# Upload your kaggle.json to /workspace/ before running this
import os, json

os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)

if os.path.exists('/workspace/kaggle.json'):
    !cp /workspace/kaggle.json ~/.kaggle/kaggle.json
    !chmod 600 ~/.kaggle/kaggle.json
    print('Kaggle API key configured from /workspace/kaggle.json')
else:
    print('WARNING: No kaggle.json found at /workspace/kaggle.json')
    print('Place your kaggle.json in the /workspace directory and re-run this cell.')

In [ ]:
# Cell 3: Download RSNA Bone Age dataset
import kagglehub

dataset_path = kagglehub.dataset_download('kmader/rsna-bone-age')
print('Dataset path:', dataset_path)

In [ ]:
# Cell 4: Load train/test CSV files
import pandas as pd
import os

training_csv_path = os.path.join(dataset_path, 'boneage-training-dataset.csv')
train_df = pd.read_csv(training_csv_path)

test_csv_path = os.path.join(dataset_path, 'boneage-test-dataset.csv')
test_df = pd.read_csv(test_csv_path)

print('Train:', train_df.shape)
print('Test:', test_df.shape)

In [ ]:
# Cell 5: Standardize column names
import numpy as np

test_df = test_df.rename(columns={'Case ID': 'id', 'Sex': 'male'})
test_df['male'] = test_df['male'].apply(lambda x: True if x == 'M' else False)

print('Train columns:', train_df.columns.tolist())
print('Test columns:', test_df.columns.tolist())

In [ ]:
# Cell 6: Add full image paths
train_image_dir = os.path.join(dataset_path, 'boneage-training-dataset', 'boneage-training-dataset')
test_image_dir = os.path.join(dataset_path, 'boneage-test-dataset', 'boneage-test-dataset')

train_df['path'] = train_df['id'].apply(lambda x: os.path.join(train_image_dir, f'{x}.png'))
test_df['path'] = test_df['id'].apply(lambda x: os.path.join(test_image_dir, f'{x}.png'))

print('Example path:', train_df['path'].iloc[0])

In [ ]:
# Cell 7: Download competition test data
!kaggle competitions download -c eliva-25-medical -p eliva_data
!unzip -q -o eliva_data/eliva-25-medical.zip -d eliva_data
print('Downloaded competition test data')

In [ ]:
# Cell 8: Load competition test CSV
comp_test_csv = 'eliva_data/test/test.csv'
comp_test_df = pd.read_csv(comp_test_csv)

if 'Case ID' in comp_test_df.columns:
    comp_test_df = comp_test_df.rename(columns={'Case ID': 'id'})
if 'Sex' in comp_test_df.columns:
    comp_test_df = comp_test_df.rename(columns={'Sex': 'male'})
    comp_test_df['male'] = comp_test_df['male'].apply(lambda x: True if x == 'M' else False)
elif 'gender' in comp_test_df.columns:
    comp_test_df['male'] = comp_test_df['gender'].apply(lambda x: True if str(x).lower() in ['m', 'male'] else False)

comp_test_image_dir = 'eliva_data/test'
comp_test_df['path'] = comp_test_df['id'].apply(lambda x: os.path.join(comp_test_image_dir, str(x)))

print('Competition test samples:', len(comp_test_df))
print(comp_test_df.head())

In [ ]:
# Cell 9: TRAINING - 3-Fold DenseNet121 Ensemble + TTA
# This is the actual training cell. It will take ~5-10 hours.
#
# NOTE: The final submission CSV is saved as 'eliva_ensemble_submission.csv'
# After training completes, download this file and submit to Kaggle manually.
#
# Pod auto-shutdown: On success OR failure, this cell stops the RunPod pod
# to prevent surprise billing. If you want it to keep running, comment out
# the 'finally' block below.

import gc, os, math, warnings, sys, traceback
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, Dense, Dropout, concatenate, GlobalAveragePooling2D, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers.schedules import CosineDecay
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator

warnings.filterwarnings('ignore')

try:
    # ===== CONFIG =====
    N_FOLDS = 3
    IMG_SIZE = 256
    BATCH_SIZE = 32
    EPOCHS_PHASE1 = 20
    EPOCHS_PHASE2 = 12

    # ===== 1. Stratified K-Fold =====
    train_df['age_group'] = pd.cut(train_df['boneage'], bins=10, labels=False)
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

    # ===== 2. Data Generators =====
    train_datagen = ImageDataGenerator(
        rescale=1./255,
        rotation_range=25,
        width_shift_range=0.15,
        height_shift_range=0.15,
        shear_range=0.15,
        zoom_range=0.15,
        brightness_range=[0.7, 1.3],
        channel_shift_range=0.15,
        fill_mode='nearest',
    )

    val_test_datagen = ImageDataGenerator(rescale=1./255)

    # Pre-load competition test images
    comp_test_generator_resized = val_test_datagen.flow_from_dataframe(
        dataframe=comp_test_df,
        x_col='path',
        y_col=None,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        class_mode=None,
        shuffle=False
    )

    comp_test_generator_resized.reset()
    all_images = []
    for i in range(math.ceil(comp_test_generator_resized.samples / BATCH_SIZE)):
        all_images.append(next(comp_test_generator_resized))
    x_images_test = np.vstack(all_images)
    x_gender_test = comp_test_df['male'].values.astype('float32')

    print(f'Test images loaded: {x_images_test.shape}')

    # ===== 3. Generator wrapper (dual inputs) =====
    def combined_generator(img_gen):
        for img_batch, targets_batch in img_gen:
            boneage_batch = targets_batch[:, 0].astype('float32')
            gender_batch = targets_batch[:, 1].astype('float32')
            yield ((img_batch, gender_batch), boneage_batch)

    # ===== 4. Build model =====
    def build_model(input_shape=(IMG_SIZE, IMG_SIZE, 3)):
        base_model = DenseNet121(weights='imagenet', include_top=False, input_shape=input_shape)

        image_input = Input(shape=input_shape, name='image_input')
        x = base_model(image_input)
        x = GlobalAveragePooling2D()(x)
        x = BatchNormalization()(x)

        gender_input = Input(shape=(1,), name='gender_input')
        combined = concatenate([x, gender_input])

        z = Dense(256, activation='relu')(combined)
        z = BatchNormalization()(z)
        z = Dropout(0.5)(z)
        z = Dense(128, activation='relu')(z)
        z = BatchNormalization()(z)
        z = Dropout(0.4)(z)
        output = Dense(1, activation='linear', name='boneage_output')(z)

        model = Model(inputs=[image_input, gender_input], outputs=output)
        return model, base_model

    # ===== 5. Train folds =====
    fold_preds = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df['age_group'])):
        print(f'\n{"="*60}')
        print(f'FOLD {fold+1}/{N_FOLDS}')
        print(f'{"="*60}')

        fold_train_df = train_df.iloc[train_idx].copy()
        fold_val_df = train_df.iloc[val_idx].copy()

        train_gen = train_datagen.flow_from_dataframe(
            dataframe=fold_train_df, x_col='path', y_col=['boneage', 'male'],
            target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
            class_mode='raw', seed=42
        )
        val_gen = val_test_datagen.flow_from_dataframe(
            dataframe=fold_val_df, x_col='path', y_col=['boneage', 'male'],
            target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
            class_mode='raw', seed=42
        )

        # --- Phase 1: Train top layers (frozen backbone) ---
        model, base_model = build_model()
        base_model.trainable = False

        lr_schedule = CosineDecay(0.001, decay_steps=(train_gen.samples // BATCH_SIZE) * EPOCHS_PHASE1)
        model.compile(optimizer=Adam(learning_rate=lr_schedule), loss='mean_absolute_error', metrics=['mean_absolute_error'])

        model.fit(
            combined_generator(train_gen),
            steps_per_epoch=train_gen.samples // BATCH_SIZE,
            epochs=EPOCHS_PHASE1,
            validation_data=combined_generator(val_gen),
            validation_steps=val_gen.samples // BATCH_SIZE,
            callbacks=[
                EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True),
                ModelCheckpoint(f'fold{fold}_p1.keras', save_best_only=True, monitor='val_loss', mode='min'),
            ],
            verbose=1
        )

        # --- Phase 2: Fine-tune all layers ---
        for layer in model.layers:
            layer.trainable = True

        model.compile(
            optimizer=Adam(learning_rate=5e-6),
            loss='mean_absolute_error',
            metrics=['mean_absolute_error']
        )

        model.fit(
            combined_generator(train_gen),
            steps_per_epoch=train_gen.samples // BATCH_SIZE,
            epochs=EPOCHS_PHASE2,
            validation_data=combined_generator(val_gen),
            validation_steps=val_gen.samples // BATCH_SIZE,
            callbacks=[
                EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
                ModelCheckpoint(f'fold{fold}_p2.keras', save_best_only=True, monitor='val_loss', mode='min'),
            ],
            verbose=1
        )

        # --- TTA Predictions ---
        best_model = load_model(f'fold{fold}_p2.keras')

        tta_preds = []

        # Original
        p = best_model.predict([x_images_test, x_gender_test], verbose=0).flatten()
        tta_preds.append(p)

        # Horizontal flip
        p = best_model.predict([np.fliplr(x_images_test), x_gender_test], verbose=0).flatten()
        tta_preds.append(p)

        # Brightness variations
        for factor in [0.85, 1.15]:
            aug = np.clip(x_images_test * factor, 0, 1)
            p = best_model.predict([aug, x_gender_test], verbose=0).flatten()
            tta_preds.append(p)

        fold_preds.append(np.mean(tta_preds, axis=0))

        # Cleanup
        del model, best_model
        gc.collect()

    # ===== 6. Ensemble all folds =====
    final_preds = np.mean(fold_preds, axis=0)
    final_preds = np.clip(final_preds, 0, 228)

    # ===== 7. Save submission CSV =====
    submission_df = pd.DataFrame({'id': comp_test_df['id'], 'boneage': final_preds})
    submission_file = 'eliva_ensemble_submission.csv'
    submission_df.to_csv(submission_file, index=False)

    print(f'\nSaved {submission_file}')
    print(submission_df.round(1))

    # Auto-submit to Kaggle competition
    import subprocess
    print('\nSubmitting to Kaggle competition...')
    result = subprocess.run(
        ['kaggle', 'competitions', 'submit', '-c', 'eliva-25-medical',
         '-f', submission_file, '-m', 'Auto-submit from RunPod training'],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        print('Submission failed. Download CSV manually before pod shuts down.')
    else:
        print('Submission successful! Check your Kaggle competition page.')

except Exception:
    traceback.print_exc()
    print('\nTRAINING FAILED. Pod will shut down in 30 seconds...')

finally:
    print('\nShutting down RunPod pod in 30 seconds...')
    print('Set RUNPOD_API_KEY in pod env vars to enable auto-stop.')
    if os.environ.get('RUNPOD_API_KEY'):
        import subprocess
        result = subprocess.run(
            ['runpodctl', 'stop', 'pod', os.environ.get('RUNPOD_POD_ID', '')],
            capture_output=True, text=True
        )
        print(result.stdout)
        if result.returncode != 0:
            print(result.stderr)
            print('Auto-stop failed. Stop the pod manually from the RunPod dashboard.')
    else:
        print('RUNPOD_API_KEY not set — please stop the pod manually from the RunPod dashboard.')
        print('(Or set RUNPOD_API_KEY as an environment variable on the pod for future runs.)')